# Congestion Rent: 2026's Most Widely Congested Moment

When was the grid most *widely* constrained in 2026 &ndash; the single moment with the most borders congested at once?

**Before you run this:** sign up at [voltpath.ai/dashboard](https://voltpath.ai/dashboard) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY`.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)).

In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from datetime import UTC, datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from common import (
    BALTIC_ZONES,
    ZONE_REGIONS,
    congestion_rent,
    fetch_flows_range,
    hourly_border_counts,
    zone_price_summary,
)

## Case study: 2026's most widely congested moments

We scan every Market Time Unit (MTU) across 2026, count how many borders are flagged `Congestion`, then take the timestamps with the highest counts.

In [ ]:
end = min(datetime.now(UTC), datetime(2027, 1, 1, tzinfo=UTC))
flows_2026 = fetch_flows_range("2026-01-01", end.strftime("%Y-%m-%d"))
counts_2026 = hourly_border_counts(flows_2026, "Congestion")
total_borders = flows_2026["border"].nunique()

max_borders = counts_2026["n_borders"].max()
tied = counts_2026[counts_2026["n_borders"] == max_borders].sort_values("datetime")
print(
    (
        f"Peak: {max_borders} of {total_borders} borders simultaneously congested, "
        f"{len(tied)} timestamp(s) tied at that max:"
    )
)
tied

## Price spike decouples Baltics + Finland from Europe

The [VoltPath dashboard set to 2026-02-03](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2026-02-03&dateTo=2026-02-04) &ndash; with the playback control dragged to 13:15 UTC &ndash; shows the first peak.

<img src="media/congestion-peak-2026-02-03-dashboard.png" alt="VoltPath dashboard showing peak simultaneous congestion on 2026-02-03" width="800">

Each snapshot will tell us a different story, and this one is best seen by looking at prices across the entire day (which VoltPath supports for any date-range selection). FI, EE, LV and LT exhibit significantly higher prices than the rest of Europe throughout.

<img src="media/congestion-2026-02-03-day-prices.gif" alt="VoltPath dashboard day-long price playback for 2026-02-03" width="800">

We can see below that the Baltic BZs and Finland landed at 3-4x of average prices elsewhere in Europe:

In [ ]:
BALTIC_FI_ZONES = BALTIC_ZONES | {"FI"}
peak_20260203 = tied["datetime"].iloc[0]
zone_price_summary(flows_2026, BALTIC_FI_ZONES, "Baltic + FI", peak_20260203)

Two factors were at play behind this isolated price spike:

1. Weather &ndash; a cold snap ranked amongst the coldest in 20 years drove heating demand up, while Finland experienced some of the lowest wind speeds in three decades. [Seinäjoki University of Applied Sciences reported](https://lehti.seamk.fi/verkkolehti/tuulen-nopeuden-poikkeuksellisuus-tammi-helmikuussa-2026/) that the average wind speed at coastal stations fell below 7 m/s.
2. Supply-side squeeze &ndash; [Estonia's Auvere power plant was down](https://news.err.ee/1609911433/auvere-power-plant-to-take-weeks-to-repair) for weeks due to a crack in the masonry lining around one of the plant's four heat exchangers, taking out ~260-270 MW of capacity from generation. Simultaneously, Elering, Estonia's Transmission State Operator, cut the Estonia->Latvia transfer capacity from 1447 MW to 922 MW in January 2026, congesting the region's import-constrained borders even further.

In [ ]:
congestion_rent(flows_2026, peak_20260203)

## Core Europe spikes while the Nordics stay flush

The [VoltPath dashboard for 2026-06-20](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2026-06-20&dateTo=2026-06-21) – with the playback control dragged to 20:15 UTC – shows that tied peak.

<img src="media/congestion-peak-2026-06-20-dashboard.png" alt="VoltPath dashboard showing peak simultaneous congestion on 2026-06-20" width="800">

In [ ]:
CORE_ZONES = {zone for zone, region in ZONE_REGIONS.items() if region == "Core"}
peak_20260620 = tied["datetime"].iloc[1]
zone_price_summary(flows_2026, CORE_ZONES, "Core", peak_20260620)

Two factors line up with this split:

1. Weather – a Central/Eastern European heatwave pushed national temperature records in Germany (41.7°C), Poland (40.5°C), Czechia (41.9°C) and Hungary (40.7°C) by late June ([Germany, Czechia, Poland and Hungary swelter through hottest days on record](https://www.theguardian.com/environment/2026/jun/28/temperature-records-tumble-across-europe-as-heatwave-moves-east)), while the Nordics and Baltics stayed mild and renewable-rich. The concrete sourcing clusters on 22–25 June rather than the 20th itself – this is the leading edge of the same heatwave, not a same-day data point.
2. Supply-side squeeze – French nuclear plants cut output as river cooling-water temperatures climbed: Golfech 2, Bugey 3 and Nogent-sur-Seine 1 shut down between 22–25 June ([Heat curbs nuclear output, France's EDF shuts down more units](https://oenergetice.cz/en/nuclear-power/heat-curbs-nuclear-output-frances-edf-shuts-down-more-units)), cutting 4.1 GW (7% of demand) on 25 June ([Reactors shut down as France's nuclear output hit by European heatwave](https://www.nucnet.org/news/reactors-shut-down-as-france-s-nuclear-output-hit-by-european-heatwave-6-5-2026)) and shrinking France's afternoon export surplus from 11–12 GW to 2.9 GW by 24 June ([When the rivers ran warm: how a heatwave thinned France's nuclear export cushion](https://www.pv-magazine.com/2026/07/01/when-the-rivers-ran-warm-how-a-heatwave-thinned-frances-nuclear-export-cushion/)) – tightening Core supply right as heat-driven demand peaked.

In [ ]:
congestion_rent(flows_2026, peak_20260620)

The [VoltPath dashboard for 2026-09-19](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2026-09-19&dateTo=2026-09-20) &ndash; with the playback control dragged to 12:45 UTC &ndash; shows that tied peak.

<img src="media/congestion-peak-2026-09-19-dashboard.png" alt="VoltPath dashboard showing peak simultaneous congestion on 2026-09-19" width="800">

In [ ]:
BALKAN_ITALY_ZONES = {
    zone for zone, region in ZONE_REGIONS.items() if region in ("South-East Europe", "Greece-Italy")
}
peak_20260919 = tied["datetime"].iloc[2]
zone_price_summary(flows_2026, BALKAN_ITALY_ZONES, "Balkans + Italy", peak_20260919)

The rest of Europe is awash in cheap power – EU-27 negative-price hours more than doubled in Q1 2026 ([Europe's negative electricity price hours double in Q1](https://www.pv-magazine.com/2026/05/08/europes-negative-electricity-price-hours-double-in-q1-amid-renewables-surpluses-market-imbalances/)) – but the Balkans/Italy cluster stays structurally tight: weaker interconnection, less wind/solar buildout, and a hydro/thermal fleet still recovering from a severe August 2026 drought that throttled Romania's Cernavodă (full shutdown from 13 August, cooling water cut off by record-low Danube levels; [ANSA](https://www.ansa.it/nuova_europa/en/news/sections/energy/2026/08/13/romania-begins-shutdown-of-nuclear-plant-due-dvia-low-danube_69797561-a288-4b4d-a5dc-d8eaac7cf9c3.html)), Hungary's Paks (cut ~90%, from 2,000 MW to ~230 MW) and Slovenia's Krško (cut 20%) for the same river-cooling reason ([Heatwave pushes power prices past €700 in SE Europe](https://balkangreenenergynews.com/heatwave-pushes-power-prices-past-eur-700-in-southeast-europe/), [Krško nuclear plant cuts output 20% on low water](https://www.devdiscourse.com/article/science-environment/3960156-slovenias-krsko-nuclear-plant-to-cut-output-by-20-due-to-low-water-levels)).

**Caveat:** none of these outage reports are dated to September – they're five to seven weeks earlier, and Cernavodă's own ~2-week restart estimate puts it plausibly back online well before 19 September. Treat the August drought as the structural reason this region runs import-dependent even in a glut, not as a confirmed same-month cause.

In [ ]:
congestion_rent(flows_2026, peak_20260919)